<a href="https://colab.research.google.com/github/champaksworldcreate/BasicHTML/blob/main/face_attendance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install django pillow numpy opencv-python face_recognition

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 8.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 29.1 MB/s eta 0:00:00


In [2]:
!rm -rf attendix
!django-admin startproject attendix
%cd attendix
!python manage.py startapp attendance

/content/attendix


In [3]:
from pathlib import Path
import textwrap

settings_path = Path("attendix/settings.py")
s = settings_path.read_text()

# Add attendance app
if "'attendance'" not in s and '"attendance"' not in s:
    s = s.replace("INSTALLED_APPS = [", "INSTALLED_APPS = [\n    'attendance',")

# Allow all hosts for tunnel URL
s = s.replace("ALLOWED_HOSTS = []", "ALLOWED_HOSTS = ['*']")

# Media settings
if "MEDIA_ROOT" not in s:
    s += "\n" + textwrap.dedent("""
    MEDIA_URL = '/media/'
    MEDIA_ROOT = BASE_DIR / 'media'
    """)

# Basic security (fine for demo; for production use proper settings)
if "CSRF_TRUSTED_ORIGINS" not in s:
    s += "\n" + "CSRF_TRUSTED_ORIGINS = ['https://*.trycloudflare.com']\n"

settings_path.write_text(s)
print("✅ settings.py updated")

✅ settings.py updated


In [4]:
from pathlib import Path
import textwrap

urls_path = Path("attendix/urls.py")
urls_path.write_text(textwrap.dedent("""
from django.contrib import admin
from django.urls import path, include
from django.conf import settings
from django.conf.urls.static import static

urlpatterns = [
    path("admin/", admin.site.urls),
    path("", include("attendance.urls")),
] + static(settings.MEDIA_URL, document_root=settings.MEDIA_ROOT)
""").strip() + "\n")

print("✅ attendix/urls.py written")

✅ attendix/urls.py written


In [5]:
from pathlib import Path
import textwrap

Path("attendance/models.py").write_text(textwrap.dedent("""
from django.db import models
from django.utils import timezone

class Person(models.Model):
    full_name = models.CharField(max_length=120)
    code = models.CharField(max_length=50, unique=True)  # roll / employee id
    is_active = models.BooleanField(default=True)

    def __str__(self):
        return f"{self.full_name} ({self.code})"

class PersonFace(models.Model):
    person = models.ForeignKey(Person, on_delete=models.CASCADE, related_name="faces")
    image = models.ImageField(upload_to="faces/")
    created_at = models.DateTimeField(auto_now_add=True)

    def __str__(self):
        return f"Face for {self.person.code} ({self.id})"

class FaceEncoding(models.Model):
    person = models.ForeignKey(Person, on_delete=models.CASCADE, related_name="encodings")
    face = models.OneToOneField(PersonFace, on_delete=models.CASCADE, related_name="encoding")
    encoding = models.JSONField()  # list[float]
    created_at = models.DateTimeField(auto_now_add=True)

    def __str__(self):
        return f"Encoding: {self.person.code} / face {self.face_id}"

class AttendancePunch(models.Model):
    person = models.ForeignKey(Person, on_delete=models.CASCADE, related_name="punches")
    date = models.DateField(default=timezone.localdate)

    in_time = models.DateTimeField(null=True, blank=True)
    out_time = models.DateTimeField(null=True, blank=True)

    in_confidence = models.FloatField(default=0.0)
    out_confidence = models.FloatField(default=0.0)

    method = models.CharField(max_length=30, default="face")
    note = models.CharField(max_length=200, blank=True, default="")

    class Meta:
        unique_together = ("person", "date")
        ordering = ["-date", "person__code"]

    def __str__(self):
        return f"{self.person.code} - {self.date}"

    @property
    def duration_seconds(self):
        if self.in_time and self.out_time:
            return int((self.out_time - self.in_time).total_seconds())
        return None
""").strip() + "\n")

print("✅ models.py written (IN/OUT punches)")

✅ models.py written (IN/OUT punches)


In [6]:
from pathlib import Path
import textwrap

Path("attendance/admin.py").write_text(textwrap.dedent("""
from django.contrib import admin
from .models import Person, PersonFace, FaceEncoding, AttendancePunch

class PersonFaceInline(admin.TabularInline):
    model = PersonFace
    extra = 1

@admin.register(Person)
class PersonAdmin(admin.ModelAdmin):
    list_display = ("code", "full_name", "is_active")
    search_fields = ("code", "full_name")
    inlines = [PersonFaceInline]

@admin.register(PersonFace)
class PersonFaceAdmin(admin.ModelAdmin):
    list_display = ("id", "person", "created_at")

@admin.register(FaceEncoding)
class FaceEncodingAdmin(admin.ModelAdmin):
    list_display = ("id", "person", "face", "created_at")

@admin.register(AttendancePunch)
class AttendancePunchAdmin(admin.ModelAdmin):
    list_display = ("person", "date", "in_time", "out_time", "in_confidence", "out_confidence", "method", "note")
    search_fields = ("person__code", "person__full_name")
""").strip() + "\n")

print("✅ admin.py written")

✅ admin.py written


In [7]:
from pathlib import Path
import textwrap

Path("attendance/face_utils.py").write_text(textwrap.dedent("""
from __future__ import annotations
import base64
import io
import numpy as np
from PIL import Image
import face_recognition

def pil_from_data_url(data_url: str) -> Image.Image:
    if "," not in data_url:
        raise ValueError("Invalid data URL")
    _, b64 = data_url.split(",", 1)
    raw = base64.b64decode(b64)
    return Image.open(io.BytesIO(raw)).convert("RGB")

def encoding_from_pil(img: Image.Image) -> np.ndarray:
    arr = np.array(img)
    locations = face_recognition.face_locations(arr, model="hog")  # fast CPU
    if not locations:
        raise ValueError("No face detected")
    encs = face_recognition.face_encodings(arr, known_face_locations=locations)
    if not encs:
        raise ValueError("Could not compute encoding")
    return encs[0]

def distance_to_confidence(distance: float) -> float:
    d = max(0.0, min(1.0, float(distance)))
    return float(max(0.0, 1.0 - d))

def best_match(
    unknown_encoding: np.ndarray,
    known_encodings: list[np.ndarray],
    known_person_ids: list[int],
    threshold: float = 0.6
):
    if not known_encodings:
        return (None, 1.0, 0.0)

    distances = face_recognition.face_distance(np.array(known_encodings), unknown_encoding)
    best_idx = int(np.argmin(distances))
    best_distance = float(distances[best_idx])
    conf = distance_to_confidence(best_distance)

    if best_distance <= threshold:
        return (known_person_ids[best_idx], best_distance, conf)

    return (None, best_distance, conf)
""").strip() + "\n")

print("✅ face_utils.py written")

✅ face_utils.py written


In [8]:
import os
from pathlib import Path
import textwrap

os.makedirs("attendance/management/commands", exist_ok=True)
Path("attendance/management/__init__.py").write_text("")
Path("attendance/management/commands/__init__.py").write_text("")

Path("attendance/management/commands/build_encodings.py").write_text(textwrap.dedent("""
from django.core.management.base import BaseCommand
from attendance.models import PersonFace, FaceEncoding
from attendance.face_utils import encoding_from_pil
from PIL import Image

class Command(BaseCommand):
    help = "Build face encodings for PersonFace images that don't have encodings."

    def handle(self, *args, **options):
        qs = PersonFace.objects.select_related("person").all()
        built, skipped, failed = 0, 0, 0

        for face in qs:
            if hasattr(face, "encoding"):
                skipped += 1
                continue

            try:
                img = Image.open(face.image.path).convert("RGB")
                enc = encoding_from_pil(img)
                FaceEncoding.objects.create(
                    person=face.person,
                    face=face,
                    encoding=[float(x) for x in enc.tolist()],
                )
                built += 1
                self.stdout.write(self.style.SUCCESS(f"Built: {face.person.code} face={face.id}"))
            except Exception as e:
                failed += 1
                self.stdout.write(self.style.ERROR(f"Failed face={face.id} ({face.person.code}): {e}"))

        self.stdout.write(self.style.SUCCESS(f"Done. built={built}, skipped={skipped}, failed={failed}"))
""").strip() + "\n")

print("✅ build_encodings command written")

✅ build_encodings command written


In [9]:
from pathlib import Path
import textwrap

Path("attendance/urls.py").write_text(textwrap.dedent("""
from django.urls import path
from . import views

urlpatterns = [
    path("", views.home, name="home"),
    path("mark/", views.mark_page, name="mark_page"),
    path("logs/", views.logs_page, name="logs_page"),
    path("report/", views.daily_report_page, name="daily_report_page"),
    path("report/csv/", views.daily_report_csv, name="daily_report_csv"),
    path("api/mark/", views.api_mark_attendance, name="api_mark_attendance"),
]
""").strip() + "\n")

Path("attendance/views.py").write_text(textwrap.dedent("""
from __future__ import annotations
import json
from datetime import date as dt_date

import numpy as np
from django.http import JsonResponse, HttpResponse
from django.shortcuts import render
from django.views.decorators.csrf import csrf_exempt
from django.utils import timezone

from .models import Person, FaceEncoding, AttendancePunch
from .face_utils import pil_from_data_url, encoding_from_pil, best_match


def home(request):
    return render(request, "attendance/home.html")


def mark_page(request):
    return render(request, "attendance/mark.html")


def logs_page(request):
    rows = AttendancePunch.objects.select_related("person").all()[:200]
    return render(request, "attendance/logs.html", {"rows": rows})


@csrf_exempt
def api_mark_attendance(request):
    if request.method != "POST":
        return JsonResponse({"ok": False, "error": "POST required"}, status=405)

    try:
        payload = json.loads(request.body.decode("utf-8"))
        data_url = payload.get("image")
        if not data_url:
            return JsonResponse({"ok": False, "error": "Missing image"}, status=400)

        threshold = float(payload.get("threshold", 0.6))

        # 1) compute encoding for captured face
        img = pil_from_data_url(data_url)
        unknown = encoding_from_pil(img)

        # 2) load known encodings
        enc_qs = FaceEncoding.objects.select_related("person").filter(person__is_active=True)

        known_encodings = []
        known_person_ids = []
        for row in enc_qs:
            known_encodings.append(np.array(row.encoding, dtype=np.float32))
            known_person_ids.append(row.person_id)

        person_id, distance, confidence = best_match(
            unknown_encoding=unknown,
            known_encodings=known_encodings,
            known_person_ids=known_person_ids,
            threshold=threshold,
        )

        if person_id is None:
            return JsonResponse({
                "ok": True,
                "matched": False,
                "distance": distance,
                "confidence": confidence,
                "message": "No match. Try better lighting or enroll more images.",
            })

        person = Person.objects.get(id=person_id)
        today = timezone.localdate()
        now = timezone.now()

        # IN/OUT logic:
        # - If no record for today => create and set IN
        # - If record exists and OUT is empty => set OUT
        # - Else => DONE
        obj, created = AttendancePunch.objects.get_or_create(
            person=person,
            date=today,
            defaults={
                "in_time": now,
                "in_confidence": confidence,
                "method": "face",
                "note": f"IN distance={distance:.4f}",
            }
        )

        action = None
        if created:
            action = "IN"
        else:
            if obj.in_time and not obj.out_time:
                obj.out_time = now
                obj.out_confidence = confidence
                obj.note = (obj.note + f" | OUT distance={distance:.4f}")[:200]
                obj.save(update_fields=["out_time", "out_confidence", "note"])
                action = "OUT"
            else:
                action = "DONE"

        return JsonResponse({
            "ok": True,
            "matched": True,
            "action": action,  # IN / OUT / DONE
            "person": {"code": person.code, "name": person.full_name},
            "distance": distance,
            "confidence": confidence,
            "message": (
                "IN marked." if action == "IN" else
                "OUT marked." if action == "OUT" else
                "Already marked IN and OUT for today."
            ),
        })

    except Exception as e:
        return JsonResponse({"ok": False, "error": str(e)}, status=400)


def daily_report_page(request):
    d = request.GET.get("date")
    if d:
        try:
            report_date = dt_date.fromisoformat(d)
        except ValueError:
            report_date = timezone.localdate()
    else:
        report_date = timezone.localdate()

    people = Person.objects.filter(is_active=True).order_by("code")
    day_map = {a.person_id: a for a in AttendancePunch.objects.filter(date=report_date)}

    rows = []
    for p in people:
        a = day_map.get(p.id)
        dur = ""
        if a and a.in_time and a.out_time:
            seconds = int((a.out_time - a.in_time).total_seconds())
            dur = f"{seconds//3600:02d}:{(seconds%3600)//60:02d}"

        rows.append({
            "code": p.code,
            "name": p.full_name,
            "present": a is not None,
            "in_time": (a.in_time if a else None),
            "out_time": (a.out_time if a else None),
            "duration": dur,
            "in_conf": (a.in_confidence if a else None),
            "out_conf": (a.out_confidence if a else None),
        })

    present_count = sum(1 for r in rows if r["present"])
    absent_count = len(rows) - present_count

    return render(request, "attendance/report.html", {
        "report_date": report_date,
        "rows": rows,
        "present_count": present_count,
        "absent_count": absent_count,
    })


def daily_report_csv(request):
    d = request.GET.get("date")
    if d:
        try:
            report_date = dt_date.fromisoformat(d)
        except ValueError:
            report_date = timezone.localdate()
    else:
        report_date = timezone.localdate()

    people = Person.objects.filter(is_active=True).order_by("code")
    day_map = {a.person_id: a for a in AttendancePunch.objects.filter(date=report_date).select_related("person")}

    resp = HttpResponse(content_type="text/csv")
    resp["Content-Disposition"] = f'attachment; filename="attendance_{report_date}.csv"'

    def esc(s):
        s = "" if s is None else str(s)
        s = s.replace('"', '""')
        return f'"{s}"'

    resp.write("Date,Code,Name,Status,InTime,OutTime,Duration,InConfidence,OutConfidence,Note\\n")

    for p in people:
        a = day_map.get(p.id)
        if a:
            dur = ""
            if a.in_time and a.out_time:
                sec = int((a.out_time - a.in_time).total_seconds())
                dur = f"{sec//3600:02d}:{(sec%3600)//60:02d}"

            status = "PRESENT" if a.in_time else "NO_IN"
            resp.write(",".join([
                esc(report_date),
                esc(p.code),
                esc(p.full_name),
                esc(status),
                esc(a.in_time),
                esc(a.out_time),
                esc(dur),
                esc(f"{a.in_confidence:.3f}" if a.in_time else ""),
                esc(f"{a.out_confidence:.3f}" if a.out_time else ""),
                esc(a.note),
            ]) + "\\n")
        else:
            resp.write(",".join([
                esc(report_date),
                esc(p.code),
                esc(p.full_name),
                esc("ABSENT"),
                esc(""), esc(""), esc(""), esc(""), esc(""), esc("")
            ]) + "\\n")

    return resp
""").strip() + "\n")

print("✅ urls.py + views.py written")

✅ urls.py + views.py written


In [11]:
from pathlib import Path
import os, textwrap

os.makedirs("attendance/templates/attendance", exist_ok=True)

base_css = """
:root { --bg:#fff7e6; --panel: rgba(255,255,255,.9); --ink:#1f2328; --muted:#6b7280; --brand:#d97706; }
body{ margin:0; font-family: system-ui, -apple-system, Segoe UI, Roboto, Arial; background:var(--bg); color:var(--ink); }
.wrap{ max-width: 1000px; margin: 0 auto; padding: 24px; }
.card{ background:var(--panel); border:1px solid rgba(0,0,0,.08); border-radius:18px; padding:18px; box-shadow: 0 10px 30px rgba(0,0,0,.08); }
.row{ display:flex; gap:10px; flex-wrap:wrap; align-items:center; }
.muted{ color:var(--muted); }
.pill{ display:inline-block; padding:8px 12px; border-radius:999px; border:1px solid rgba(0,0,0,.1); background:rgba(255,255,255,.6); font-weight:800; text-decoration:none; color:inherit; }
.btn{ display:inline-block; padding:10px 14px; border-radius:12px; background:var(--brand); color:#fff; text-decoration:none; font-weight:900; border:0; cursor:pointer; }
.btn2{ background:#111827; }
code{ background:rgba(0,0,0,.06); padding:2px 6px; border-radius:8px; }
"""

Path("attendance/templates/attendance/home.html").write_text(textwrap.dedent(f"""
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width,initial-scale=1" />
  <title>Attendix — Face Attendance</title>
  <style>{base_css}</style>
</head>
<body>
  <div class="wrap">
    <div class="card">
      <h1 style="margin:0 0 6px">Attendix</h1>
      <p class="muted" style="margin:0 0 14px">Face-recognition attendance with IN/OUT and daily reports.</p>

      <div class="row" style="margin-bottom:14px">
        <a class="btn" href="/mark/">Mark (IN / OUT)</a>
        <a class="btn btn2" href="/report/">Daily Report</a>
        <a class="btn" href="/logs/" style="background:#4b5563">Recent Marks</a>
        <a class="btn" href="/admin/" style="background:#065f46">Admin</a>
      </div>

      <hr style="border:none;border-top:1px solid rgba(0,0,0,.08); margin:16px 0;">

      <div class="muted">
        <div><b>Enroll:</b> create a Person in Admin, upload 2–5 clear face photos.</div>
        <div style="margin-top:6px"><b>Then run:</b> <code>python manage.py build_encodings</code></div>
        <div style="margin-top:10px">
          <b>Daily flow:</b> first scan marks <b>IN</b>, second scan marks <b>OUT</b>.
        </div>
      </div>
    </div>
  </div>
</body>
</html>
""").strip()+"\n")

Path("attendance/templates/attendance/mark.html").write_text(textwrap.dedent(f"""
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width,initial-scale=1" />
  <title>Mark IN/OUT — Attendix</title>
  <style>
    {base_css}
    .grid{{ display:grid; gap:14px; grid-template-columns: 1.2fr .8fr; }}
    @media (max-width: 900px){{ .grid{{ grid-template-columns:1fr; }} }}
    video, canvas{{ width:100%; border-radius:14px; background:#111827; }}
    input{{ width:100%; padding:10px 12px; border-radius:12px; border:1px solid rgba(0,0,0,.12); }}
    .out{{ white-space:pre-wrap; font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 13px;
      background:rgba(0,0,0,.04); padding:10px; border-radius:12px; border:1px solid rgba(0,0,0,.08); }}
  </style>
</head>
<body>
  <div class="wrap">
    <div class="card">
      <div class="row" style="justify-content:space-between">
        <div>
          <h1 style="margin:0 0 6px">Mark IN / OUT</h1>
          <div class="muted">Allow camera access, then capture.</div>
        </div>
        <div class="row">
          <a class="pill" href="/">Home</a>
          <a class="pill" href="/report/">Report</a>
          <a class="pill" href="/admin/">Admin</a>
        </div>
      </div>

      <div class="grid" style="margin-top:14px">
        <div>
          <video id="v" autoplay playsinline></video>
          <canvas id="c" style="display:none"></canvas>

          <div class="row" style="margin-top:10px">
            <button class="btn" id="btnCapture">Capture & Submit</button>
            <button class="btn btn2" id="btnRetry">Retry Camera</button>
          </div>

          <p class="muted" style="margin:10px 0 0">
            Tips: face centered, good lighting, reduce motion blur.
          </p>
        </div>

        <div>
          <label class="muted" style="display:block;margin-bottom:6px;font-weight:900">Match Threshold</label>
          <input id="threshold" type="number" step="0.01" min="0.30" max="0.80" value="0.60" />

          <div style="height:10px"></div>
          <div class="out" id="out">Ready.</div>

          <div style="height:12px"></div>
          <a class="pill" href="/logs/">Recent Marks</a>
        </div>
      </div>
    </div>
  </div>

<script>
(function(){{
  const video = document.getElementById("v");
  const canvas = document.getElementById("c");
  const out = document.getElementById("out");
  const thresholdEl = document.getElementById("threshold");
  let stream = null;

  function log(msg){{ out.textContent = msg; }}

  async function startCamera(){{
    try{{
      if(stream){{ stream.getTracks().forEach(t => t.stop()); stream = null; }}}
      stream = await navigator.mediaDevices.getUserMedia({{
        video: {{ facingMode: "user", width: {{ ideal: 1280 }}, height: {{ ideal: 720 }} }},
        audio: false
      }});
      video.srcObject = stream;
      log("Camera started. Click “Capture & Submit”.");
    }}catch(e){{
      log("Camera error: " + e.message);
    }}
  }}

  async function captureAndSend(){{
    if(!stream){{ log("Camera not started."); return; }}}
    const w = video.videoWidth || 1280;
    const h = video.videoHeight || 720;
    canvas.width = w; canvas.height = h;
    const ctx = canvas.getContext("2d");
    ctx.drawImage(video, 0, 0, w, h);

    const dataUrl = canvas.toDataURL("image/jpeg", 0.85);
    log("Sending snapshot for recognition...");

    try{{
      const res = await fetch("/api/mark/", {{
        method: "POST",
        headers: {{ "Content-Type": "application/json" }},
        body: JSON.stringify({{
          image: dataUrl,
          threshold: parseFloat(thresholdEl.value || "0.6")
        }})
      }});
      const j = await res.json();

      if(!j.ok){{
        log("Error: " + (j.error || "Unknown"));
        return;
      }}
      if(!j.matched){{
        log(
          "No match.\n" +
          "distance=" + (j.distance?.toFixed?.(4) ?? j.distance) + "\n" +
          "confidence=" + (j.confidence?.toFixed?.(3) ?? j.confidence) + "\n" +
          (j.message || "")
        );
        return;
      }}

      const action = j.action || "UNKNOWN";
      log(
        "Result ✅\n" +
        "action=" + action + "\n" +
        "code=" + j.person.code + "\n" +
        "name=" + j.person.name + "\n" +
        "distance=" + (j.distance?.toFixed?.(4) ?? j.distance) + "\n" +
        "confidence=" + (j.confidence?.toFixed?.(3) ?? j.confidence) + "\n\n" +
        (j.message || "")
      );
    }}catch(e){{
      log("Request failed: " + e.message);
    }}
  }}

  document.getElementById("btnRetry").addEventListener("click", startCamera);
  document.getElementById("btnCapture").addEventListener("click", captureAndSend);

  if(!navigator.mediaDevices || !navigator.mediaDevices.getUserMedia){{
    log("Camera APIs not supported in this browser.");
  }} else {{
    startCamera();
  }}
}})();
</script>
</body>
</html>
""").strip()+"\n")

Path("attendance/templates/attendance/logs.html").write_text(textwrap.dedent(f"""
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width,initial-scale=1" />
  <title>Recent Marks — Attendix</title>
  <style>
    {base_css}
    table{{ width:100%; border-collapse:collapse; margin-top:12px; }}
    th, td{{ padding:10px 8px; border-bottom:1px solid rgba(0,0,0,.08); text-align:left; }}
    th{{ color:var(--muted); font-size:12px; letter-spacing:.04em; text-transform:uppercase; }}
  </style>
</head>
<body>
  <div class="wrap">
    <div class="card">
      <div class="row" style="justify-content:space-between">
        <h1 style="margin:0">Recent Marks</h1>
        <div class="row">
          <a class="pill" href="/">Home</a>
          <a class="pill" href="/mark/">Mark</a>
          <a class="pill" href="/report/">Report</a>
        </div>
      </div>

      <table>
        <thead>
          <tr>
            <th>Date</th>
            <th>Code</th>
            <th>Name</th>
            <th>IN</th>
            <th>OUT</th>
            <th>IN Conf</th>
            <th>OUT Conf</th>
            <th>Note</th>
          </tr>
        </thead>
        <tbody>
          {% for r in rows %}
          <tr>
            <td>{{ r.date }}</td>
            <td>{{ r.person.code }}</td>
            <td>{{ r.person.full_name }}</td>
            <td>{{ r.in_time|default_if_none:"" }}</td>
            <td>{{ r.out_time|default_if_none:"" }}</td>
            <td>{{ r.in_confidence|floatformat:3 }}</td>
            <td>{{ r.out_confidence|floatformat:3 }}</td>
            <td>{{ r.note }}</td>
          </tr>
          {% empty %}
          <tr><td colspan="8">No entries yet.</td></tr>
          {% endfor %}
        </tbody>
      </table>
    </div>
  </div>
</body>
</html>
""").strip()+"\n")

Path("attendance/templates/attendance/report.html").write_text(textwrap.dedent(f"""
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width,initial-scale=1" />
  <title>Daily Attendance Report — Attendix</title>
  <style>
    {base_css}
    table{{ width:100%; border-collapse:collapse; margin-top:12px; }}
    th, td{{ padding:10px 8px; border-bottom:1px solid rgba(0,0,0,.08); text-align:left; }}
    th{{ color:var(--muted); font-size:12px; letter-spacing:.04em; text-transform:uppercase; }}
    input{{ padding:10px 12px; border-radius:12px; border:1px solid rgba(0,0,0,.14); }}
    .present{{ color:#065f46; font-weight:900; }}
    .absent{{ color:#b91c1c; font-weight:900; }}
  </style>
</head>
<body>
  <div class="wrap">
    <div class="card">
      <div class="row" style="justify-content:space-between">
        <div>
          <h1 style="margin:0 0 6px">Daily Attendance Report</h1>
          <div class="row">
            <span class="pill">Date: {{ report_date }}</span>
            <span class="pill">Present: {{ present_count }}</span>
            <span class="pill">Absent: {{ absent_count }}</span>
          </div>
        </div>
        <div class="row">
          <a class="pill" href="/">Home</a>
          <a class="pill" href="/mark/">Mark</a>
          <a class="pill" href="/admin/">Admin</a>
        </div>
      </div>

      <div class="row" style="margin-top:14px">
        <form method="get" action="/report/" class="row">
          <input type="date" name="date" value="{{ report_date|date:'Y-m-d' }}" />
          <button class="pill" type="submit">Open</button>
        </form>
        <a class="pill" href="/report/csv/?date={{ report_date|date:'Y-m-d' }}">Download CSV</a>
      </div>

      <table>
        <thead>
          <tr>
            <th>Code</th>
            <th>Name</th>
            <th>Status</th>
            <th>IN</th>
            <th>OUT</th>
            <th>Duration</th>
          </tr>
        </thead>
        <tbody>
          {% for r in rows %}
          <tr>
            <td>{{ r.code }}</td>
            <td>{{ r.name }}</td>
            <td>
              {% if r.present %}
                <span class="present">PRESENT</span>
              {% else %}
                <span class="absent">ABSENT</span>
              {% endif %}
            </td>
            <td>{{ r.in_time|default_if_none:"" }}</td>
            <td>{{ r.out_time|default_if_none:"" }}</td>
            <td>{{ r.duration }}</td>
          </tr>
          {% empty %}
          <tr><td colspan="6">No people found.</td></tr>
          {% endfor %}
        </tbody>
      </table>
    </div>
  </div>
</body>
</html>
""").strip()+"\n")

print("✅ templates written")

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1997235756.py, line 119)

In [ ]:
!python manage.py makemigrations
!python manage.py migrate
print("✅ migrations done")

print("\nNow create an admin user (choose username/password):")
!python manage.py createsuperuser

In [ ]:
!python manage.py runserver 0.0.0.0:8000 &
print("✅ Django running on port 8000")

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import subprocess, re, time

p = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(200):
    line = p.stdout.readline().strip()
    if line:
        m = re.search(r"(https://[a-zA-Z0-9\\-]+\\.trycloudflare\\.com)", line)
        if m:
            public_url = m.group(1)
            break
    time.sleep(0.05)

if public_url:
    print("✅ Public URL:", public_url)
    print("Admin:", public_url + "/admin/")
    print("Mark:", public_url + "/mark/")
    print("Report:", public_url + "/report/")
else:
    print("❌ Could not detect the URL. Scroll the output above and find the trycloudflare link.")